# Round 5 — ML Research Notebook

Per-family ML research entrypoint. Reuses `round5.research_lib` for raw data/microstructure and `round5.ml.*` for panel/labels/models/gate.

**Scope locked**: forward-return + toxicity targets, linear+tree models only, per-family, no trader.py integration.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'round5').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from round5 import research_lib as rl
from round5.ml import ml_features as mf
from round5.ml import ml_models as mm
print('available models:', mm.available_models())

## 1. Build the feature frame for one family

In [ ]:
FAMILY = 'PEBBLES'
df = mf.build_feature_frame(FAMILY)
print(f'panel shape: {df.shape}')
print(f'products: {sorted(df["product"].unique())}')
feats = mf.feature_columns(df)
print(f'n features: {len(feats)}')
df.head()

## 2. Quick Ridge IC across horizons (smoke test)

In [ ]:
ic_table = []
for h in (20, 50, 100):
    y = mf.build_labels(df, target='fwd_ret', horizon=h)
    X = df[feats]
    keep = X.notna().all(axis=1) & y.notna()
    X, y, df_a = X[keep], y[keep], df[keep]
    for fold, tr, te in mm.cv_walkforward(df_a):
        y_pred, _ = mm.fit_predict('ridge', X[tr.values], y[tr.values], X[te.values])
        ic = mm.information_coefficient(y_pred, y[te.values].values)
        ic_table.append({'horizon': h, 'fold': fold, 'ic': ic})
pd.DataFrame(ic_table)

## 3. LightGBM full fit + feature importance (D2+D3 -> D4)

In [ ]:
h = 50
y = mf.build_labels(df, target='fwd_ret', horizon=h)
X = df[feats]
keep = X.notna().all(axis=1) & y.notna()
X, y, df_a = X[keep], y[keep], df[keep]
tr = df_a['day'].isin([2, 3])
te = df_a['day'] == 4
y_pred, model = mm.fit_predict('lgbm', X[tr.values], y[tr.values], X[te.values])
ic = mm.information_coefficient(y_pred, y[te.values].values)
print(f'LGBM h={h} IC = {ic:.4f}')
fi = mm.feature_importance('lgbm', model, feats).head(20)
fi

## 4. Tradeability gate walkthrough

In [ ]:
df_te = df_a[te.values].reset_index(drop=True)
gate = mm.tradeability_gate(df_te, y_pred, horizon=h, live_haircut=0.3)
import json
print(json.dumps(gate, indent=2, default=float))

## 5. Cross-family summary — run after `ml_family_report.py --family ALL`

Aggregate `round5/reports/<FAMILY>/ml/metrics.csv` across families to build a tradeability matrix.

In [ ]:
rows = []
for fam in rl.FAMILIES:
    p = ROOT / 'round5' / 'reports' / fam / 'ml' / 'metrics.csv'
    if p.exists():
        m = pd.read_csv(p)
        m['family'] = fam
        rows.append(m)
if rows:
    cross = pd.concat(rows, ignore_index=True)
    headline = cross[cross['fold'] == 'D2+D3->D4']
    pivot = headline.pivot_table(index=['family', 'target'], columns=['horizon', 'model'], values='ic')
    pivot
else:
    print('no ml reports yet — run ml_family_report.py first')